# Explorer — Vietnamese Legal Documents EDA

Deep exploratory data analysis of `lovienal/vietnamese-legal-documents` processed through the hfdata pipeline.

## 0. Setup & Load

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

figure_dir = Path("figures")
figure_dir.mkdir(exist_ok=True)

latex_template = go.layout.Template()
latex_template.layout = go.Layout(
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family="Computer Modern, CMU Serif, STIX Two Text, Latin Modern Roman, serif"),
    xaxis=dict(showgrid=True, gridcolor="lightgray", gridwidth=1, zeroline=False),
    yaxis=dict(showgrid=True, gridcolor="lightgray", gridwidth=1, zeroline=False),
    coloraxis=dict(colorscale="piyg", colorbar=dict(outlinewidth=0)),
    colorscale=dict(sequential="piyg", sequentialminus="piyg", diverging="piyg"),
)
pio.templates["latex_white"] = latex_template
pio.templates.default = "plotly_white+latex_white"

BASE = Path("datasets/lovienal/vietnamese-legal-documents")
# MODEL_ID = "nvidia/llama-embed-nemotron-8b"
# MODEL_ID = "nvidia/llama-nemotron-embed-1b-v2"
# MODEL_ID = "GreenNode/GreenNode-Embedding-Large-VN-Mixed-V1"
MODEL_ID = "microsoft/harrier-oss-v1-27b"
# ── Load metadata (has category fields) ──────────────────────────────────────
meta_dir = BASE / "metadata" / "data" / "preprocessed"
meta_df = pd.concat(
    [pd.read_parquet(f) for f in sorted(meta_dir.glob("*.parquet"))],
    ignore_index=True,
)

# Expand metadata_json into flat columns
meta_base = meta_df.drop(columns=["metadata_json"]).reset_index(drop=True)
meta_expanded = pd.json_normalize(meta_df["metadata_json"].apply(json.loads))
meta_expanded = meta_expanded.drop(
    columns=[c for c in meta_expanded.columns if c in meta_base.columns], errors="ignore"
)
df = pd.concat([meta_base, meta_expanded], axis=1)

# Parse dates
df["issuance_date"] = pd.to_datetime(
    df["issuance_date"], format="%d/%m/%Y", errors="coerce"
)
df["year"] = df["issuance_date"].dt.year.astype("Int64")

# ── Load reduced embeddings (aligned by row order) ──────────────────────────
red_dir = BASE / "content" / "data" / "embreduced" / MODEL_ID
red_df = pd.concat(
    [pd.read_parquet(f) for f in sorted(red_dir.glob("*.parquet"))],
    ignore_index=True,
)

print(f"Metadata rows : {len(df):,}")
print(f"Reduced rows  : {len(red_df):,}")
print(f"Columns (meta): {list(df.columns)}")
print(f"Columns (red) : {list(red_df.columns)}")

## 1. Metadata Schema Study

In [ ]:
schema_info = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notnull().sum(),
    "null_pct": (df.isnull().sum() / len(df) * 100).round(1),
    "unique": df.nunique(),
    "sample": df.iloc[0].astype(str).str[:80],
})
schema_info

In [ ]:
# ── Disk sizes ───────────────────────────────────────────────────────────────
def dir_size_mb(path: Path) -> float:
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1e6

sizes = {}
for stage in ["raw", "preprocessed"]:
    for config in ["metadata", "content"]:
        p = BASE / config / "data" / stage
        if p.exists():
            sizes[f"{config}/{stage}"] = dir_size_mb(p)
for sub in ["embeddings", "embreduced"]:
    p = BASE / "content" / "data" / sub / MODEL_ID
    if p.exists():
        sizes[f"content/{sub}"] = dir_size_mb(p)

print(f"Total records: {len(df):,}\n")
print("Disk usage (MB):")
for k, v in sizes.items():
    print(f"  {k:30s} {v:>10,.1f} MB")
print(f"  {'TOTAL':30s} {sum(sizes.values()):>10,.1f} MB")

# ── Text length distribution ─────────────────────────────────────────────────
df["text_len"] = df["text"].str.len()

fig = px.histogram(
    df, x="text_len", nbins=100,
    title="Document Title Length Distribution",
    labels={"text_len": "Character count"},
    marginal="box",
)
fig.update_layout(height=600)
fig.show()
fig.write_image(figure_dir / "Document Title Length Distribution.png", scale=2)

print(f"\nText length stats:")
print(df["text_len"].describe().to_string())

## 3. Category Distributions

In [ ]:
# ── Legal type ────────────────────────────────────────────────────────────────
type_counts = df["legal_type"].value_counts().head(20)
fig = px.bar(
    x=type_counts.values, y=type_counts.index, orientation="h",
    title="Top 20 Legal Types",
    labels={"x": "Count", "y": "Legal Type"},
)
fig.update_layout(height=600, yaxis=dict(autorange="reversed"))
fig.show()
fig.write_image(figure_dir / "Top 20 Legal Types.png", scale=2)


In [ ]:
# ── Legal sectors ────────────────────────────────────────────────────────────
sector_counts = df["legal_sectors"].value_counts().head(20)
fig = px.bar(
    x=sector_counts.values, y=sector_counts.index, orientation="h",
    title="Top 20 Legal Sectors",
    labels={"x": "Count", "y": "Legal Sector"},
)
fig.update_layout(height=600, yaxis=dict(autorange="reversed"))
fig.show()
fig.write_image(figure_dir / "Top 20 Legal Sectors.png", scale=2)


In [ ]:
# ── Issuing authority ────────────────────────────────────────────────────────
auth_counts = df["issuing_authority"].value_counts().head(20)
fig = px.bar(
    x=auth_counts.values, y=auth_counts.index, orientation="h",
    title="Top 20 Issuing Authorities",
    labels={"x": "Count", "y": "Issuing Authority"},
)
fig.update_layout(height=600, yaxis=dict(autorange="reversed"))
fig.show()
fig.write_image(figure_dir / "Top 20 Issuing Authorities.png", scale=2)


In [ ]:
# ── Year distribution ─────────────────────────────────────────────────────────
year_counts = df["year"].dropna().astype(int).value_counts().sort_index()
fig = px.bar(
    x=year_counts.index, y=year_counts.values,
    title="Documents per Year",
    labels={"x": "Year", "y": "Count"},
)
fig.update_layout(height=600)
fig.show()
fig.write_image(figure_dir / "Documents per Year.png", scale=2)


In [ ]:
# ── Cross-tab heatmap: legal_type x year ─────────────────────────────────────
top_types = df["legal_type"].value_counts().head(10).index
ct = pd.crosstab(
    df[df["legal_type"].isin(top_types)]["legal_type"],
    df[df["legal_type"].isin(top_types)]["year"].dropna().astype(int),
)
# Keep only years with data
ct = ct.loc[:, ct.sum() > 0]

fig = px.imshow(
    ct, aspect="auto",
    title="Legal Type x Year (Top 10 Types)",
    labels=dict(x="Year", y="Legal Type", color="Count"),
    color_continuous_scale="BuPu",
)
fig.update_layout(height=600)
fig.show()
fig.write_image(figure_dir / "Legal Type x Year (Top 10 Types).png", scale=2)


## 4. 2D Embedding Scatter (Interactive)

Toggle **algorithm** (PCA / t-SNE / UMAP) and **color** (legal type / sector / issuing authority) via dropdowns.

In [ ]:
# Merge metadata with reduced embeddings (row-aligned)
n = min(len(df), len(red_df))
viz = df.iloc[:n].reset_index(drop=True).copy()
for col in red_df.columns:
    viz[col] = red_df[col].values[:n]

# Limit legal_type / legal_sectors / issuing_authority to top-15 + "Other"
for col in ["legal_type", "legal_sectors", "issuing_authority"]:
    top = viz[col].value_counts().head(15).index
    viz[f"{col}_top"] = viz[col].where(viz[col].isin(top), "Other")

# ── Subsample for performance (518K is too heavy for browser) ────────────────
SAMPLE_N = 50_000
if len(viz) > SAMPLE_N:
    viz_sample = viz.sample(n=SAMPLE_N, random_state=42).reset_index(drop=True)
    print(f"Subsampled to {SAMPLE_N:,} points for interactive plots")
else:
    viz_sample = viz

ALGORITHMS = {
    "PCA":   ("pca_2d_x",  "pca_2d_y"),
    "t-SNE": ("tsne_2d_x", "tsne_2d_y"),
    "UMAP":  ("umap_2d_x", "umap_2d_y"),
}

COLOR_FIELDS = {
    "Legal Type":        "legal_type_top",
    "Legal Sector":      "legal_sectors_top",
    "Issuing Authority": "issuing_authority_top",
}

fig = go.Figure()

combo_map = {}
idx = 0
for algo_name, (xcol, ycol) in ALGORITHMS.items():
    for color_name, color_col in COLOR_FIELDS.items():
        for cat in sorted(viz_sample[color_col].unique()):
            mask = viz_sample[color_col] == cat
            fig.add_trace(go.Scattergl(
                x=viz_sample.loc[mask, xcol],
                y=viz_sample.loc[mask, ycol],
                mode="markers",
                marker=dict(size=2, opacity=0.6),
                name=str(cat),
                visible=False,
                text=viz_sample.loc[mask, "text"].str[:80],
                hovertemplate="%{text}<extra>%{fullData.name}</extra>",
            ))
            combo_map.setdefault((algo_name, color_name), []).append(idx)
            idx += 1

default_key = ("PCA", "Legal Type")
for i in combo_map.get(default_key, []):
    fig.data[i].visible = True

algo_list = list(ALGORITHMS.keys())
color_list = list(COLOR_FIELDS.keys())

buttons_combined = []
for algo_name in algo_list:
    for color_name in color_list:
        visible = [False] * len(fig.data)
        for i in combo_map.get((algo_name, color_name), []):
            visible[i] = True
        buttons_combined.append(dict(
            label=f"{algo_name}  \u00b7  {color_name}",
            method="update",
            args=[{"visible": visible}, {"title": f"2D Embeddings \u2014 {algo_name} \u00b7 {color_name}"}],
        ))

fig.update_layout(
    title=f"2D Embeddings \u2014 PCA \u00b7 Legal Type",
    height=600,
    margin=dict(l=60, r=250, t=80, b=60),
    width=1000,
    xaxis=dict(domain=[0, 0.78], automargin=False),
    yaxis=dict(domain=[0, 1], automargin=False),
    updatemenus=[
        dict(
            buttons=buttons_combined,
            direction="down",
            x=1.0, xanchor="left", y=1.0, yanchor="top",
            showactive=True,
            active=0,
        ),
    ],
    annotations=[
        dict(text="View:", x=1.0, xref="paper", y=1.06, yref="paper",
             showarrow=False, xanchor="left"),
    ],
    legend=dict(itemsizing="constant", x=0.80, xanchor="left", y=0.75, yanchor="top"),
)
fig.show()

for _algo in algo_list:
    for _color in color_list:
        _vis = [False] * len(fig.data)
        for _i in combo_map.get((_algo, _color), []):
            _vis[_i] = True
        for _j, _v in enumerate(_vis):
            fig.data[_j].visible = _v
        fig.layout.title.text = f"2D Embeddings - {_algo} - {_color}"
        fig.write_image(figure_dir / f"2D Embeddings - {_algo} - {_color}.png", scale=2)
